In [4]:
import pyemu
import os
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning) 
import pandas as pd
import matplotlib.pyplot as plt
import psutil
import shutil
import numpy as np
import sys
import swatmf

In [5]:
swatmf.__version__

'1.0.1'

# 01. Set working directory

In [6]:
# path to project directory
prj_dir = "C:\\Users\\seonggpa\\Documents\\projects\\watersheds\\hbasnet_opt"
main_opt_path = os.path.join(prj_dir, 'main_opt')
os.chdir(main_opt_path)

# 02. Create prior pst

In [13]:
# create prior pst based on reweigted control file
pst = pyemu.Pst('koki_zon_rw.pst')

In [14]:
# set prior UA
pst.pestpp_options['ies_drop_conflicts'] = True
pst.pestpp_options['ies_no_noise'] = True
pst.pestpp_options['ies_num_reals'] = 300 # number of realization
pst.control_data.noptmax = -1 # number of iteration
pst.model_command = 'python forward_run.py'
pst.write('koki_zon_rw_prior.pst', version=2) # write new IES control file

noptmax:-1, npar_adj:50, nnz_obs:13706


In [ ]:
os.chdir(os.pardir)

In [16]:
os.getcwd()

'D:\\spark\\koksilah'

# 04. Run IES

## 04.1 Set up IES

In [17]:
# check number of cores on your computer
num_workers = psutil.cpu_count(logical=False)

In [18]:
# name for prior ua directory
m_d = os.path.join(os.getcwd(), "koki_zon_rw_prior")

## 04.2 Execute

In [ ]:
pyemu.os_utils.start_workers(main_opt_path, # the folder which contains the "template" PEST dataset
                            'pestpp-ies', #the PEST software version we want to run
                            'koki_zon_rw_prior.pst', # the control file to use with PEST
                            num_workers=num_workers, #how many agents to deploy
                            worker_root='.', #where to deploy the agent directories; relative to where python is running
                            master_dir=m_d, #the manager directory,
                            # reuse_master=True
                            )

# 05. Analyze results
## 05.1 Check model performance

In [ ]:
pst = pyemu.Pst(os.path.join(m_d,'mb_zon_rw_ies.pst')) # load control file

In [ ]:
# load prior simulation
pr_oe = pyemu.ObservationEnsemble.from_csv(
    pst=pst,filename=os.path.join(m_d,"mb_zon_rw_ies.0.obs.csv")
    )
# load posterior simulation
pt_oe = pyemu.ObservationEnsemble.from_csv(pst=pst,filename=os.path.join(m_d,"mb_zon_rw_ies.{0}.obs.csv".format(pst.control_data.noptmax)))


In [ ]:
# plot 1 to 1 scatter plot and residuals
pyemu.plot_utils.res_1to1(pst);

In [ ]:
# check phi values
pt_oe.phi_vector

In [ ]:
# plot progress
fig,ax = plt.subplots(1,1)
pr_oe.phi_vector.apply(np.log10).hist(ax=ax,fc="0.5",ec="none",alpha=0.5,density=False)
pt_oe.phi_vector.apply(np.log10).hist(ax=ax,fc="b",ec="none",alpha=0.5,density=False)
_ = ax.set_xlabel("$log_{10}\\phi$")

## 05.2 Prior and Posterior parameter probability

In [ ]:
prior_df = pyemu.ParameterEnsemble.from_csv(pst=pst,filename=os.path.join(m_d,"mb_zon_rw_ies.{0}.par.csv".format(0)))
post_df = pyemu.ParameterEnsemble.from_csv(pst=pst,filename=os.path.join(m_d,"mb_zon_rw_ies.{0}.par.csv".format(pst.control_data.noptmax)))

In [ ]:
df_pars = pd.read_csv(os.path.join(m_d, "mb_zon_rw_ies.par_data.csv"))
sel_pars = df_pars.loc[df_pars["partrans"]=='log']
sel_pars

In [ ]:
swatmf_viz.plot_prior_posterior_par_hist(prior_df, post_df, sel_pars)

## 05.3 Predictive Uncertainty

In [ ]:
swatmf_viz.plot_tseries_ensembles(pst, pr_oe, pt_oe, height=6, dot=True)

In [ ]:
os.getcwd()

In [ ]:
pst.parrep(parfile=os.path.join(m_d, "mb_zon_rw_ies.{0}.base.csv".format(pst.control_data.noptmax)))

In [ ]:
# updates the model input files with parameter values
pst.write_input_files(pst_path=m_d)

In [ ]:
# run the model forward run; this applies all the SWAT and MODFLOW paarameters, executes SWAT-MODFLOW 
os.chdir(m_d)
pyemu.os_utils.run('python forward_run.py')